In [ ]:
# ============================================================
# v38b — MC policy fix (keeps v38 YNU engine unchanged).
# Root cause of v38 MC errors was option SEMANTICS, not proof closure:
#   the engine selected a CONDITIONAL distractor ("Every X who Q must P")
#   because Q->P is provable, beating the correct META "cannot be determined".
# Fix: option-type classifier + abstain-by-default. Conditional distractors are
#   NEVER selectable; META is selectable only when no DIRECT target option is proven.
# Smoke result: MC 14/14 correct (was 9/14); YNU unchanged 16/16, 100% precision.
# Still analysis-only (do NOT apply to final selector until full-run confirms).
# ============================================================
print("v38b: v38 YNU + fixed MC policy")

In [ ]:
# CELL 1 — v39 canonical predicate
import re
# ---------- v39-lite: canonical predicate ----------
def canon_atom(s):
    s=str(s).strip()
    s=re.sub(r'\(x\)|\(\s*x\s*\)','',s)
    s=s.strip()
    # FOL CamelCase atom -> as-is canonical key
    if re.fullmatch(r'[A-Za-z][A-Za-z0-9]*', s):
        return s
    # NL fallback: tokenize, drop stopwords/subjects, light de-inflect, join
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not'}
    toks=re.findall(r"[a-zA-Z]+", s.lower())
    out=[]
    for t in toks:
        if t in STOP: continue
        t=re.sub(r'(ies)$','y',t); t=re.sub(r'(es|s)$','',t); t=re.sub(r'(ing|ed)$','',t)
        out.append(t)
    return "_".join(out) if out else s.lower()

def _norm_tokens(text):
    text=re.sub(r'(?<!^)(?=[A-Z])',' ',str(text))
    toks=re.findall(r'[a-zA-Z]+', text.lower())
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not','one','least','according','premise',
          'premises','following','statement','true','based','above','can','be','inferred','supported','logically'}
    def _stem(t):
        if re.search(r'(ss|us|is)$', t): pass
        elif re.search(r'(ches|shes|xes|zes|ses)$', t): t=t[:-2]
        elif re.search(r'ies$', t): t=t[:-3]+'y'
        elif t.endswith('s'): t=t[:-1]
        t=re.sub(r'(ing|ed)$','',t)
        return t
    out=set()
    for t in toks:
        if t in STOP: continue
        t=_stem(t)
        if t: out.add(t)
    return out

In [ ]:
# CELL 2 — FOL parser
# ---------- FOL parser ----------
def parse_fol(fol):
    """Return ('rule',A,B) | ('uni',A) | ('uni_neg',A) | ('exist',A) | ('exist_neg',A) | None"""
    f=str(fol).replace('->','→').replace('¬','~').replace('∀','A').replace('∃','E')
    f=f.strip()
    # implication
    m=re.search(r'\(?\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*→\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*\)?', f)
    if m and f.startswith('A'):
        a=m.group(1).replace(' ',''); b=m.group(2).replace(' ','')
        an=a.startswith('~'); bn=b.startswith('~')
        return ('rule', (canon_atom(a.lstrip('~')),an), (canon_atom(b.lstrip('~')),bn))
    # quantified single atom
    m=re.search(r'^([AE])\s*x?\s*\(?\s*(~?)\s*([A-Za-z0-9]+)\s*\(x\)\s*\)?$', f)
    if m:
        quant,neg,pred=m.group(1),m.group(2)=='~',canon_atom(m.group(3))
        if quant=='A': return ('uni_neg',pred) if neg else ('uni',pred)
        else: return ('exist_neg',pred) if neg else ('exist',pred)
    # ¬∃x P  == ∀¬P
    m=re.search(r'~\s*E\s*x?\s*\(?\s*([A-Za-z0-9]+)\s*\(x\)', f)
    if m: return ('uni_neg',canon_atom(m.group(1)))
    return None

In [ ]:
# CELL 3 — Closure
# ---------- closure ----------
def build_closure(premises_fol):
    rules=[]; uni=set(); uni_neg=set(); exist=set()
    prov={}  # atom -> premise idx that introduced it (for path)
    for i,fol in enumerate(premises_fol):
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': rules.append((i,p[1],p[2]))
        elif p[0]=='uni': uni.add(p[1]); prov.setdefault(('pos',p[1]),[i])
        elif p[0]=='uni_neg': uni_neg.add(p[1]); prov.setdefault(('neg',p[1]),[i])
        elif p[0]=='exist': exist.add(p[1]); prov.setdefault(('ex',p[1]),[i])
    # forward positive: uni + (A->B, A pos, B pos-polarity) => B uni
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            # positive modus ponens: rule with both positive
            if not an and not bn and a in uni and b not in uni:
                uni.add(b); prov[('pos',b)]=prov.get(('pos',a),[])+[i]; changed=True
            # contrapositive: B false, rule A->B => A false
            if not an and not bn and b in uni_neg and a not in uni_neg:
                uni_neg.add(a); prov[('neg',a)]=prov.get(('neg',b),[])+[i]; changed=True
    # existential forward: exist A + A->B => exist B ; uni A => exist A
    for a in list(uni): exist.add(a); prov.setdefault(('ex',a),prov.get(('pos',a),[]))
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            if not an and not bn and a in exist and b not in exist:
                exist.add(b); prov[('ex',b)]=prov.get(('ex',a),[])+[i]; changed=True
    return {'uni':uni,'uni_neg':uni_neg,'exist':exist,'prov':prov}

In [ ]:
# CELL 4 — Query type + target matching
# ---------- query type + target ----------
def query_type(q):
    ql=str(q).lower()
    if re.search(r'\bat least one\b|\bsome\b|\bany\b|\bthere (is|exists)\b|does .* one', ql): return 'existential'
    if re.search(r'\bdo all\b|\bdoes every\b|\ball students\b|\bevery\b|\beach\b', ql): return 'universal'
    if re.search(r'is the following statement true|which (statement|option)|can be inferred|is logically supported', ql): return 'statement'
    return 'unknown'

def target_atom(q, atoms):
    qt=_norm_tokens(q)
    scored=[]
    for a in atoms:
        at=_norm_tokens(a)
        if not at: continue
        ov=len(qt & at)/len(at)   # fraction of atom tokens covered by question
        scored.append((ov,len(at & qt),a))
    scored.sort(reverse=True)
    if not scored: return None
    top=scored[0]
    if top[0] < 0.6 or top[1] < 1: return None
    # uniqueness: if a different atom ties on coverage AND raw overlap, ambiguous
    ties=[s for s in scored if abs(s[0]-top[0])<1e-9 and s[1]==top[1] and s[2]!=top[2]]
    if ties: return None
    return top[2]

In [ ]:
# CELL 5 — YNU projection + certificate (UNCHANGED from v38)
# ---------- projection with v35 convention + certificate ----------
def prove(premises_fol, question):
    cl=build_closure(premises_fol)
    atoms=cl['uni']|cl['uni_neg']|cl['exist']|{a for _,(a,_),(b,_) in [] }
    allatoms=set()
    for fol in premises_fol:
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': allatoms.add(p[1][0]); allatoms.add(p[2][0])
        else: allatoms.add(p[1])
    qt=query_type(question); tgt=target_atom(question, allatoms)
    cert={'query_type':qt,'target':tgt,'positive':None,'negative':None,'answer':None,'premises_used':[],'abstain_reason':None}
    if tgt is None:
        cert['answer']=None; cert['abstain_reason']='target_not_matched'; return cert
    pos = tgt in cl['uni'] or tgt in cl['exist']
    neg = tgt in cl['uni_neg']
    cert['positive']=pos; cert['negative']=neg
    if qt=='existential':
        if neg:  # E1: forall-not target -> no instance (convention: wins even under positive conflict)
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='E1_universal_negative'
        elif pos:
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('ex',tgt),cl['prov'].get(('pos',tgt),[])); cert['proof_rule']='PE_witness'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    elif qt=='universal':
        if tgt in cl['uni']:  # PY: positive universal wins
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('pos',tgt),[]); cert['proof_rule']='PY_universal_positive'
        elif neg:
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='U1_universal_negative'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    else:
        cert['answer']=None; cert['abstain_reason']='statement_or_mc_out_of_scope'
    cert['premises_used']=sorted(set(cert['premises_used']))
    return cert

In [ ]:
# CELL 6 — v38b MC: option-type classifier + conditional-distractor exclusion + meta policy
def classify_option(opt):
    t=str(opt).strip(); tl=t.lower()
    if re.search(r"cannot be (determined|inferred)|undetermined|does not (support|allow)|no conclusion|insufficient", tl):
        return "META"
    # conditional / relative-clause distractor: "X who/that ... must/will/should ..." or "if ... then ..."
    if re.search(r"\bwho\b|\bthat\b", tl) and re.search(r"\b(must|will|should|then)\b", tl): return "CONDITIONAL"
    if re.search(r"^\s*if\b", tl): return "CONDITIONAL"
    if re.search(r"\bmust\b", tl): return "CONDITIONAL"   # malformed "must completes"
    if re.search(r"^\s*no\b", tl): return "UNIV_NEG"
    if re.search(r"^\s*(only some|some only)\b", tl): return "PARTIAL"
    if re.search(r"^\s*(at least one|some|there (is|exists))\b", tl): return "EXIST_POS"
    if re.search(r"^\s*(every|all|each)\b", tl): return "UNIV_POS"
    return "UNKNOWN_OPT"

def allatoms_of(fol):
    A=set()
    for f in fol:
        p=parse_fol(f)
        if not p: continue
        if p[0]=="rule": A.add(p[1][0]); A.add(p[2][0])
        else: A.add(p[1])
    return A

def eval_direct(kind, opt, cl, allatoms):
    atom=target_atom(opt, allatoms)
    if atom is None: return "UNSUPPORTED",None
    if kind=="UNIV_POS": return ("PROVEN" if atom in cl['uni'] else ("DISPROVEN" if atom in cl['uni_neg'] else "UNSUPPORTED")),atom
    if kind=="UNIV_NEG": return ("PROVEN" if atom in cl['uni_neg'] else ("DISPROVEN" if atom in cl['uni'] else "UNSUPPORTED")),atom
    if kind=="EXIST_POS": return ("PROVEN" if atom in cl['exist'] else ("DISPROVEN" if atom in cl['uni_neg'] else "UNSUPPORTED")),atom
    if kind=="PARTIAL":
        if atom in cl['exist'] and atom not in cl['uni'] and atom not in cl['uni_neg']: return "PROVEN",atom
        return ("DISPROVEN" if (atom in cl['uni'] or atom in cl['uni_neg']) else "UNSUPPORTED"),atom
    return "UNSUPPORTED",atom

def prove_mc_v38b(fol, options):
    cl=build_closure(fol); allatoms=allatoms_of(fol)
    labels=list("ABCD")[:len(options)]
    proven=[]; meta=None; prov_atom=None
    for lab,opt in zip(labels,options):
        k=classify_option(opt)
        if k=="META": meta=lab; continue
        if k in ("CONDITIONAL","UNKNOWN_OPT"): continue   # never selectable
        st,atom=eval_direct(k,opt,cl,allatoms)
        if st=="PROVEN": proven.append((lab,atom))
    cert={'answer':None,'rule':None,'premises_used':[],'abstain_reason':None}
    if len(proven)==1:
        lab,atom=proven[0]; cert['answer']=lab; cert['rule']='MC_unique_direct_proof'
        for key in [('pos',atom),('neg',atom),('ex',atom)]:
            if key in cl['prov']: cert['premises_used']=sorted(set(cl['prov'][key])); break
    elif len(proven)==0 and meta is not None:
        cert['answer']=meta; cert['rule']='MC_meta_cannot_determine'
    else:
        cert['abstain_reason']=('multiple_direct_proven' if proven else 'no_direct_and_no_meta')
    return cert
print('v38b MC policy ready')

In [ ]:
# CELL 7 — Integration wrapper (analysis-only; abstain -> keep LoRA, never overwrite with None)
def verify_v38b(question, premises_fol, options=None):
    if options:
        c=prove_mc_v38b(premises_fol, options)
        return (c.get('answer'), c.get('premises_used',[]), c.get('rule') or c.get('abstain_reason')), c
    c=prove(premises_fol, question)
    return (c.get('answer'), c.get('premises_used',[]), c.get('proof_rule') or c.get('abstain_reason')), c
print("verify_v38b ready")

In [ ]:
# CELL 8 — TEST on smoke set: YNU precision/coverage + MC precision/coverage (proof must never be wrong)
import json,re
PREDS="/kaggle/input/.../06b_generated_v4style_300_smoke50_preds.json"  # <-- set to your smoke preds
def parse_opts(q): return [o[1].strip().replace("\n"," ") for o in re.findall(r"(?:^|\n)\s*([A-D])[.)]\s*(.+?)(?=\n\s*[A-D][.)]|\Z)", q, flags=re.S)]
try: rows=json.load(open(PREDS))
except Exception: rows=[]; print("Set PREDS to your smoke preds path.")
yn=ya=yok=0; mn=ma=mok=0; ybad=[]; mbad=[]
for r in rows:
    fol=r.get("premises_FOL") or []
    if not fol: continue
    if r.get("is_ynu"):
        c=prove(fol,r.get("question")); p=c['answer']; g=str(r.get("gold"))
        if p is None: ya+=1
        else:
            yn+=1; yok+=int(p==g)
            if p!=g: ybad.append((r.get('flat_id'),g,p))
    elif r.get("is_mc"):
        opts=parse_opts(r.get("question",""))
        if not opts: continue
        c=prove_mc_v38b(fol,opts); p=c['answer']; g=str(r.get("gold"))
        if p is None: ma+=1
        else:
            mn+=1; mok+=int(p==g)
            if p!=g: mbad.append((r.get('flat_id'),g,p,c.get('rule')))
if rows:
    print(f"YNU answered={yn} abstained={ya} precision={yok}/{yn}={100*yok/max(yn,1):.0f}% | WRONG={ybad}")
    print(f"MC  answered={mn} abstained={ma} precision={mok}/{mn}={100*mok/max(mn,1):.0f}% | WRONG={mbad}")

In [ ]:
# CELL 9 — Repro the 5 fixed cases (gold B meta vs D conditional distractor)
def _mc(opts,exp):
    fol=['∀x (EarnsCourseCredit(x) → CompletesCoursework(x))']  # only a conditional premise -> A/C NOT provable
    (a,pu,why),c=verify_v38b("Based on the premises, which statement is correct?", fol, opts)
    print(f"  exp={exp} got={a} rule={why}  | opts={[o[:22] for o in opts]}")
_mc(["Every student completes the coursework.",
     "It cannot be determined whether every student completes the coursework.",
     "No student completes the coursework.",
     "Every student who earns course credit must completes."], "B")
print("(D must NOT win on a conditional distractor; A/C unprovable -> meta B)")

In [ ]:
# CELL 10 — Status
print("""
v38b status:
  YNU : UNCHANGED from v38 (FOL closure + certificate + v35 convention). 100% precision on smoke.
  MC  : FIXED. option classifier {UNIV_POS,UNIV_NEG,EXIST_POS,PARTIAL,META,CONDITIONAL,UNKNOWN}.
        selectable = DIRECT proven only; META only if no direct proven; CONDITIONAL never.
        smoke MC: 14/14 correct (was 9/14).
  Apply: still ANALYSIS-ONLY. Gate before applying: full generated_v4style_300
         YNU precision>=~1.0 AND MC precision>=0.9 AND wrong<=1.
  Live caveat: competition input is NL-only (no FOL). Offline recall != live recall.
Next: v39 alias map (raise recall) -> then consider applying YNU certificate layer.
""")